# DINO dataset-fit exploration — checkpoint 1

We are studying **pretrained DINO patch features**, starting with DINOv2-B/14 and later repeating the same inputs with DINOv3-B/16. No model is trained in this study.

The team has already validated and inspected the dataset. Here we only prepare a fixed DINO study sample and its paired inputs; we do not rerun dataset-quality audits.

**This checkpoint:** environment → select matched trios → freeze 448×448 crops → preview one trio. Run the four code cells below in order, then stop. Model loading and heatmaps come in the next checkpoint.

Select the project's **Python 3.11 `.venv` kernel** first; setup commands are in [README.md](README.md). The kernel readout should report CPU on this laptop. Package/model downloads are never triggered by these cells.

**Study design:** one regression per operator from the committed training split, with benign and no-change controls from the same page and at the same location. Locations are chosen using ground truth, so this is guided feature-sensitivity exploration, not full-page detection evaluation. One example per operator supports tentative observations only.

**Presentation relevance:** this checkpoint fixes the methodology and the exact inputs for the backbone comparison. It makes no claims about which model or mutation type is easier.


In [ ]:
# 1. Show exactly which environment will run the experiment.
# Expected: Python 3.11, torch 2.8.x+cpu, device=cpu on this laptop.
import importlib.metadata as metadata
import json
import platform
import sys
from pathlib import Path

import matplotlib
import numpy as np
import scipy
import torch
from IPython.display import display

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
versions = {
    name: metadata.version(name)
    for name in ('torch', 'torchvision', 'transformers', 'numpy', 'scipy',
                 'Pillow', 'matplotlib', 'ipykernel')
}
environment = {
    'python': sys.version,
    'executable': sys.executable,
    'platform': platform.platform(),
    'packages': versions,
    'device': str(DEVICE),
    'torch_cuda_version': torch.version.cuda,
}
if DEVICE.type == 'cuda':
    environment['gpu'] = torch.cuda.get_device_name(DEVICE)
    environment['gpu_memory_gib'] = round(
        torch.cuda.get_device_properties(DEVICE).total_memory / 1024**3, 2
    )
print(json.dumps(environment, indent=2))
print('NumPy, SciPy and PyTorch imports succeeded. No DINO model is loaded yet.')


In [ ]:
# 2. Use existing metadata and the official split; never create a new split.
# Expected: 13 matched trios, each with regression/benign/no_change.
import random
from collections import defaultdict

REPO_ROOT = next(
    (p for p in (Path.cwd(), *Path.cwd().parents)
     if (p / 'pyproject.toml').is_file() and (p / 'src/regression_lens').is_dir()),
    None,
)
if REPO_ROOT is None:
    raise RuntimeError('Open the notebook from inside this repository.')

from regression_lens.data.records import load_records, operator_names
from regression_lens.data.splits import TEAM_SPLIT, load_split

DATA_ROOT = Path(r'D:\csc409-data\d2c_v3')  # Change this on the lab machine.
OUTPUT_ROOT = REPO_ROOT / 'outputs' / 'dino_fit'
SEED, CROP_SIZE = 490, 448
SPLIT_PATH = REPO_ROOT / TEAM_SPLIT
records = load_records(DATA_ROOT)
split = load_split(SPLIT_PATH, records)
train_pages = set(split['train'])
records_by_id = {r['sample_id']: r for r in records}
by_page = defaultdict(list)
for record in records:
    if record['parent_id'] in train_pages:
        by_page[record['parent_id']].append(record)

rng = random.Random(SEED)
trios, used_pages = [], set()
for index, operator in enumerate(operator_names(records)):
    candidates = sorted(
        (r for rows in by_page.values() for r in rows
         if r['label'] == 'regression' and r['operator'] == operator
         and all(any(c['label'] == label for c in by_page[r['parent_id']])
                 for label in ('benign', 'no_change'))),
        key=lambda r: r['sample_id'],
    )
    rng.shuffle(candidates)
    if not candidates:
        raise RuntimeError(f'No matched trio available for {operator}; do not silently omit it.')
    regression = next((r for r in candidates if r['parent_id'] not in used_pages), candidates[0])
    controls = by_page[regression['parent_id']]
    browser = ('firefox', 'webkit')[index % 2]
    benign = sorted(
        (r for r in controls if r['label'] == 'benign'),
        key=lambda r: (r.get('browser_cand') != browser, r['sample_id']),
    )[0]
    no_change = min((r for r in controls if r['label'] == 'no_change'),
                    key=lambda r: r['sample_id'])
    trios.append({'operator': operator, 'parent_id': regression['parent_id'],
                  'sample_ids': [r['sample_id'] for r in (regression, benign, no_change)]})
    used_pages.add(regression['parent_id'])

for trio in trios:
    print(f"{trio['operator']:18} {trio['parent_id']} | " + ', '.join(trio['sample_ids']))
print(f'\n{len(trios)} trios / {3 * len(trios)} pair-crop evaluations; train pages only.')


In [ ]:
# 3. Freeze model-independent crop geometry, without resizing either image.
# Expected: a reusable sample_manifest.json under ignored outputs/dino_fit/.
import hashlib
import math
from PIL import Image

def crop_rectangle(regression):
    boxes = [regression.get(key) for key in ('bbox_before', 'bbox_after')]
    boxes = [box for box in boxes if box is not None]
    if not boxes:
        raise ValueError(f"{regression['sample_id']}: edited-element boxes unavailable")
    left = min(box[0] for box in boxes)
    top = min(box[1] for box in boxes)
    right = max(box[0] + box[2] for box in boxes)
    bottom = max(box[1] + box[3] for box in boxes)
    width, height = regression['size_ref']
    x = max(0, min(math.floor((left + right - CROP_SIZE) / 2), max(0, width - CROP_SIZE)))
    y = max(0, min(math.floor((top + bottom - CROP_SIZE) / 2), max(0, height - CROP_SIZE)))
    return [x, y, x + CROP_SIZE, y + CROP_SIZE]

def load_crop(relative_path, rectangle):
    # Every paired image uses this same original-coordinate rectangle.
    # Keep a validity mask so later statistics can ignore white padding.
    x0, y0, x1, y1 = rectangle
    canvas = Image.new('RGB', (x1 - x0, y1 - y0), 'white')
    valid = np.zeros((y1 - y0, x1 - x0), dtype=bool)
    with Image.open(DATA_ROOT / relative_path) as source:
        rgb = source.convert('RGB')
        right, bottom = min(x1, rgb.width), min(y1, rgb.height)
        if right > x0 and bottom > y0:
            canvas.paste(rgb.crop((x0, y0, right, bottom)), (0, 0))
            valid[:bottom - y0, :right - x0] = True
    return canvas, valid

for trio in trios:
    trio['crop_xyxy'] = crop_rectangle(records_by_id[trio['sample_ids'][0]])
manifest = {
    'format_version': 1, 'purpose': 'ground-truth-guided DINO feature-fit exploration',
    'seed': SEED, 'subset': 'train', 'crop_size': CROP_SIZE,
    'split_sha256': hashlib.sha256(SPLIT_PATH.read_bytes()).hexdigest(),
    'metadata_sha256': hashlib.sha256(
        (DATA_ROOT / 'master_dataset_with_benign.json').read_bytes()).hexdigest(),
    'trios': trios,
}
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
manifest_path = OUTPUT_ROOT / 'sample_manifest.json'
if manifest_path.exists():
    if json.loads(manifest_path.read_text(encoding='utf-8')) != manifest:
        raise RuntimeError('Existing study differs. Preserve it and choose a new OUTPUT_ROOT.')
else:
    manifest_path.write_text(json.dumps(manifest, indent=2) + '\n', encoding='utf-8')
(OUTPUT_ROOT / 'environment.json').write_text(
    json.dumps(environment, indent=2) + '\n', encoding='utf-8')
print(f'Frozen sample: {manifest_path}')
print('Crop size:', CROP_SIZE, 'x', CROP_SIZE, '; resize: none; missing image area: white padding.')


In [ ]:
# 4. Preview exactly what DINO will receive for one matched trio.
# This is a model-input preview, not another general dataset inspection.
# Orange/cyan: edited element before/after. Red dashed: diff_bbox.
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle

PREVIEW_OPERATOR = 'font_size'  # Later try another operator from the printed list.
trio = next(t for t in trios if t['operator'] == PREVIEW_OPERATOR)
regression, benign, no_change = [records_by_id[sid] for sid in trio['sample_ids']]
rectangle = trio['crop_xyxy']
image_specs = [
    (regression, 'reference_path', 'Reference'),
    (regression, 'candidate_path', 'Regression'),
    (benign, 'candidate_path', f"Benign ({benign.get('browser_cand', 'browser')})"),
    (no_change, 'candidate_path', 'No change'),
]
fig, axes = plt.subplots(1, 4, figsize=(16, 5))
for ax, (record, path_key, title) in zip(axes, image_specs):
    image, valid = load_crop(record[path_key], rectangle)
    ax.imshow(image)
    ax.set_title(f'{title}\n{valid.mean():.1%} real image area')
    ax.axis('off')
    boxes = []
    if title == 'Reference':
        boxes = [('bbox_before', 'orange', '-')]
    elif record['label'] == 'regression':
        boxes = [('bbox_after', 'cyan', '-'), ('diff_bbox', 'red', '--')]
    for key, color, style in boxes:
        if record.get(key) is not None:
            x, y, width, height = record[key]
            ax.add_patch(Rectangle((x - rectangle[0], y - rectangle[1]), width, height,
                                   fill=False, edgecolor=color, linestyle=style, linewidth=2))
    ax.set_xlim(-0.5, CROP_SIZE - 0.5)
    ax.set_ylim(CROP_SIZE - 0.5, -0.5)
fig.suptitle(f"{PREVIEW_OPERATOR} | {trio['parent_id']} | crop {rectangle}")
fig.tight_layout()
preview_path = OUTPUT_ROOT / f'input_trio_{PREVIEW_OPERATOR}.png'
fig.savefig(preview_path, dpi=150, bbox_inches='tight')
plt.show()
print('Pair labels:', [r['label'] for r in (regression, benign, no_change)])
print('Crop image shape: (448, 448, 3); next checkpoint makes [1, 3, 448, 448] tensors.')
print(f'Preview saved to: {preview_path}')
print('STOP: inspect the crop, edited-element coverage and controls before adding DINOv2 cells.')
